
# Minimal Epoch State Exporter (CSV Only, No Delegation Fields)

This notebook runs the simulation and saves **only** per-epoch snapshots of agents to CSV—
**no outcomes**, and **no delegation-related columns**.

**Outputs (`csv_out/`):**
- `dreps_state.csv`: `[epoch, drep_id, opinion, stake]`
- `delegators_state.csv`: `[epoch, delegator_id, opinion, stake, s]`

You can later reconstruct delegations from these CSVs in a separate script.


In [42]:

%load_ext autoreload
%autoreload 2

import os, sys, random
from pathlib import Path
import pandas as pd

# Ensure local utils.py is importable
cwd = Path().resolve()
assert (cwd / "utils.py").exists(), "utils.py not found in current directory."
sys.path.insert(0, str(cwd))

from utils import DRep, Delegator, World  # uses your existing classes


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [43]:

# --- Simulation parameters ---
N_DREPS       = 100
N_DELEGATORS  = 2000
EPOCHS        = 10
SHIFT_X       = 0.1   # uniform right shift of DRep opinions each epoch (cap at 1). Set 0.0 to disable.
SEED          = 421

OUT_DIR = Path("csv_out_exp")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Writing CSVs to:", OUT_DIR.resolve())


Writing CSVs to: /Users/Joel/Documents/GitHub/ada_drep/simulation/csv_out_exp


In [44]:

from math import exp

rng = random.Random(SEED)

def sample_opinion_drep(rng):
    return rng.betavariate(4,4)

def sample_opinion_delegator(rng):
    return rng.betavariate(4,4)

def sample_stake(rng):
    return rng.random()

def sample_stickiness(rng, mean=0.6, k=40):
    # a = mean * k
    # b = (1 - mean) * k
    # return rng.betavariate(a, b)
    return rng.random()


In [45]:

# Initialize DReps
dreps = [DRep(id=f"d{k+1}", opinion=sample_opinion_drep(rng), stake=sample_stake(rng))
         for k in range(N_DREPS)]

# Initialize Delegators; initial delegation may exist internally, but we don't export it
delegators = []
for k in range(N_DELEGATORS):
    op = sample_opinion_delegator(rng)
    s  = sample_stickiness(rng)  # in [0,1]
    st = sample_stake(rng)
    # initial current can be None; or assign closest if you want internal dynamics — doesn't affect exports
    delegators.append(Delegator(id=f"a{k+1}", opinion=op, stake=st, s=s, current=None))

world = World(dreps=dreps, delegators=delegators, rng=rng)
print(f"Initialized world with {len(world.dreps)} DReps and {len(world.delegators)} delegators.")


Initialized world with 100 DReps and 2000 delegators.


In [46]:

dreps_rows = []
deleg_rows = []

for epoch in range(EPOCHS):
    # Optional: run world dynamics (redelegation rules). This updates internal state
    # but does not affect the exported columns (since we omit delegation fields).
    world.epoch()

    # Snapshot rows (no Wprime, no current_drep_id)
    for d in world.dreps:
        dreps_rows.append({
            "epoch": epoch,
            "drep_id": d.id,
            "opinion": d.opinion,
            "stake": d.stake,
        })

    for a in world.delegators:
        deleg_rows.append({
            "epoch": epoch,
            "delegator_id": a.id,
            "opinion": a.opinion,
            "stake": a.stake,
            "s": a.s,
        })

    # Optional: shift DRep opinions for next epoch
    if SHIFT_X > 0.0:
        for d in world.dreps:
            d.opinion = min(1.0, d.opinion + SHIFT_X)

# Write CSVs
dreps_df  = pd.DataFrame(dreps_rows)
deleg_df  = pd.DataFrame(deleg_rows)

dreps_path = OUT_DIR / "dreps_state.csv"
deleg_path = OUT_DIR / "delegators_state.csv"

dreps_df.to_csv(dreps_path, index=False)
deleg_df.to_csv(deleg_path, index=False)

print("Saved:", dreps_path.resolve())
print("Saved:", deleg_path.resolve())


delegetor a2 maintained due to stickiness
delegetor a5 maintained due to stickiness
delegetor a10 maintained due to stickiness
delegetor a15 maintained due to stickiness
delegetor a17 maintained due to stickiness
delegetor a18 maintained due to stickiness
delegetor a19 maintained due to stickiness
delegetor a22 maintained due to stickiness
delegetor a26 maintained due to stickiness
delegetor a28 maintained due to stickiness
delegetor a30 maintained due to stickiness
delegetor a33 maintained due to stickiness
delegetor a34 maintained due to stickiness
delegetor a35 maintained due to stickiness
delegetor a38 maintained due to stickiness
delegetor a39 maintained due to stickiness
delegetor a41 maintained due to stickiness
delegetor a42 maintained due to stickiness
delegetor a43 maintained due to stickiness
delegetor a45 maintained due to stickiness
delegetor a47 maintained due to stickiness
delegetor a48 maintained due to stickiness
delegetor a54 maintained due to stickiness
delegetor a55